# 节点 2：跑完预训练、用 KMeans 初始化好簇心、并完成 ML/CL 索引池的抽样，400 轮聚类迭代的 for epoch 循环（DSSC.py:338）

In [1]:
# 获取先前保存的变量
import scanpy as sc
import numpy as np
import scipy.sparse as sp

adata = sc.read_h5ad("./data/adata.h5ad")

A   = sp.load_npz("./data/A.npz")
A_n = sp.load_npz("./data/A_n.npz")

d = np.load("./data/arrays.npz", allow_pickle=True)  # 含 p_ 时要 allow_pickle
count_X = d["count_X"]
pos_ = d["pos_"]
y = d["y"]
p_ = d["p_"]
p_ = [np.asarray(x, dtype=int) for x in d["p_"]]
ml_ind1 = d["ml_ind1"]
ml_ind2 = d["ml_ind2"]
cl_ind1 = d["cl_ind1"]
cl_ind2 = d["cl_ind2"]
n_clusters = int(d["n_clusters"])

/home/liuzy/Software/miniconda3/envs/dssc-4090/lib/python3.8/site-packages/setuptools_scm/_integration/setuptools.py:31: RuntimeWarning: 
ERROR: setuptools==59.8.0 is used in combination with setuptools_scm>=8.x

Your build configuration is incomplete and previously worked by accident!
setuptools_scm requires setuptools>=61

Suggested workaround if applicable:
 - migrating from the deprecated setup_requires mechanism to pep517/518
   and using a pyproject.toml to declare build dependencies
   which are reliably pre-installed before running the build tools

  warnings.warn(


In [3]:
import sys, pathlib
sys.path.append(str(pathlib.Path("../../src").resolve()))
from DSSC import DSSC

# 创建模型对象
model = DSSC(input_dim=adata.n_vars, encodeLayer=[128], decodeLayer=[128], encodeHead=3, 
            encodeConcat=True, gamma=0.01, activation="elu", z_dim=32, sigma = 0.1, device="cuda").to("cuda")
#! 注意 encodeConcat 参数，DSSC 里存的值是 True，但真正起作用的等价于 False。因为 DSSC.py:93 构造编码器时没有把 encodeConcat 传进去，GATEncoder 始终用默认 concat=False
# 若问初始化时给 DSSC 传入的值，是 True；若问实际构建出的模型按哪个分支走，是 False。

## 1. 预训练 train_model（run_DSSC.py:147 → DSSC.py:148-178）

### 进入该模块的数据：
| 数据 | 来源（节点 1 处） | 变量名 | 位置                                            |
| --- | ---------------- | ----- | ---------------------------------------------- |
| adata.X (4221×2000, log 归一化) | h5 X → HVG | X_v | DSSC.py:149,155 |
| count_X (4221×2000, 原始 counts) | h5 X → 同 HVG 列 | X_raw_v | DSSC.py:152,158 |
| size_factors (4221,) | h5 X 行和 | size_factor_v | DSSC.py:153,159 |
| A_n (4221×4221, 归一化邻接) | h5 Pos → kNN=20 → norm_adj | dgl 图 G | DSSC.py:150 |
| A (dense 邻接) | 同上 | A_v（预训练里未使用） | DSSC.py:157 |
| ml / cl 索引 | — | 本阶段完全不参与 | — |

In [4]:
# run_DSSC 中步骤
from time import time

print(str(model))
t0 = time()
model.train_model(adata.X, A_n, A, count_X, adata.obs.size_factors, lr=0.001, train_iter=200, verbose=True)
print('Pre-training time: %d seconds.' % int(time() - t0))
# 下一个 python cell 为对该步骤的拆解

DSSC(
  (encoder): GATEncoder(
    (layers): ModuleList(
      (0): GATConv(
        (fc): Linear(in_features=2000, out_features=384, bias=False)
        (feat_drop): Dropout(p=0.0, inplace=False)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (leaky_relu): LeakyReLU(negative_slope=0.2)
      )
      (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ELU(alpha=1.0)
    )
    (enc_mu): GATConv(
      (fc): Linear(in_features=128, out_features=96, bias=False)
      (feat_drop): Dropout(p=0, inplace=False)
      (attn_drop): Dropout(p=0, inplace=False)
      (leaky_relu): LeakyReLU(negative_slope=0.2)
    )
  )
  (decoder): Sequential(
    (0): Linear(in_features=32, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ELU(alpha=1.0)
  )
  (_dec_mean): Sequential(
    (0): Linear(in_features=128, out_features=2000, bias=True)
    (1): MeanAct()
  )
  (_dec_d

### 一次前向的数据流（DSSC.py:165 → aeForward DSSC.py:112-121）
1. 加噪：x_c = X_v + 0.1·randn（DSSC.py:113，sigma=0.1），构成去噪自编码器输入。
2. GAT 编码器（DSSC.py:114）：z = encoder(G, x_c)
- 由于 encodeConcat 未被传入（节点 1 记录的问题），实际走 concat=False 分支（DSSC.py:70-75）：
    - GATConv(2000→128, 3 heads)：沿 A_n 把每个细胞与它的 20 个空间邻居做图注意力聚合，输出 (4221,3,128)；
    - torch.sum(dim=1) 合并 3 个头 → (4221,128)；随后 BatchNorm(128) 被连续调用两次、ELU 被跳过（DSSC.py:74-75）；
- enc_mu = GATConv(128→32, 3 heads) → sum heads → z0/z (4221,32)。
- 关键点：这里的图卷积使隐表示融合了空间邻域信息，而不是逐细胞独立编码。
3. 解码器（DSSC.py:115）：x_ = decoder(z)，即 Linear(32→128)+BN+ELU（DSSC.py:94）→ (4221,128)。
4. 三个输出头（DSSC.py:95-97,116-118）：
- mean = exp(clamp(Linear(128→2000)))（MeanAct，layers.py:239）
- disp = softplus(clamp(Linear(128→2000)))（DispAct，layers.py:247）
- pi = sigmoid(Linear(128→2000))（零膨胀概率）
- 每个形状 (4221,2000)。
5. 返回 z0, mean, disp, pi（DSSC.py:121）；注意第 120 行又用干净输入 X_v 跑了一遍编码器，得到 z0 供后续聚类用（多一次前向）。

In [ ]:
# DSSC.py:149~161，位于 1 步骤前，以下为拆 train_model
import torch
import dgl
from torch.autograd import Variable
import torch.optim as optim
# 类型/格式转换。把 numpy 的 X、X_raw（原始 counts）、size_factor 转成 torch.tensor；
# 并用 dgl.from_scipy(A_n) 把归一化邻接 A_n 建成 DGL 图 G，再把 X 设为图节点特征 G.ndata['feat']
# 这样图注意力编码器才能做消息传递
X = torch.tensor(adata.X, dtype=torch.float32)
G = dgl.from_scipy(A_n)
G.ndata['feat'] = X
X_raw = torch.tensor(count_X, dtype=torch.float32)
size_factor = torch.tensor(adata.obs.size_factors, dtype=torch.float32)
# 把张量包成 Variable 并搬到 GPU
X_v = Variable(X).to("cuda")
G_v = G.to("cuda")
X_raw_v = Variable(X_raw).to("cuda")
size_factor_v = Variable(size_factor).to("cuda")
# 细胞数
num = X.shape[0]
# 建 Adam 优化器（amsgrad=True），只优化 requires_grad 的参数
optim_adam = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001, amsgrad=True)
# 为 ZINB 去噪自编码器预训练（163行起的循环）搭建数据、图和优化器，本身不做任何计算

In [ ]:
# 步骤 1
# 仅展示单个循环，前向一次
model.train() # 切到训练模式，BatchNorm 用当前 batch 统计量
# _, mean_tensor, disp_tensor, pi_tensor = model.aeForward(G_v, X_v)

# 进入 aeForward 函数
x_c = X_v + torch.randn_like(X_v) * 0.1 # 加噪，DSSC.py:113

# 步骤 2 ~ 5
z = model.encoder(G_v, x_c) #! z：由加噪的表达矩阵（x_c）和邻接关系（G_v）得到的含噪隐表示（由损坏输入得到的 32 维 latent）
x_ = model.decoder(z) # 解码器输出：decoder(z) = Linear(32→128)+BN+ELU
mean = model._dec_mean(x_) # ZINB 的均值参数（MeanAct=exp(clamp(...))），注意其输入是x_，解码器输出
disp = model._dec_disp(x_) # ZINB 的离散度参数（DispAct=softplus(clamp(...))）
pi = model._dec_pi(x_) # ZINB 的零膨胀概率（sigmoid，dropout 概率）
z0 = model.encoder(G_v, X_v) #! z0：干净隐表示：encoder(G_v, X_v)，用未加噪输入再前向一次（DSSC.py:120），供后续聚类用
_, mean_tensor, disp_tensor, pi_tensor = z0, mean, disp, pi

In [12]:
print(z.shape)
print(x_.shape)

print(mean.shape) # [4221, 2000]
print(disp.shape) # [4221, 2000]
print(pi.shape) # [4221, 2000]
# 对第 i 个细胞、第 j 个基因）：
# mean[i,j]：该基因在该细胞上的期望表达量（负二项的均值 μ）。由 MeanAct 保证为正。
# disp[i,j]：离散度/方差参数 θ，控制负二项分布的"方差比均值大多少"（过离散程度）。由 DispAct=softplus 保证为正。
# pi[i,j]：零膨胀概率，即这个观测值为 0 是"结构性 dropout"（本来就不表达）而非"没测到"的概率。由 sigmoid 保证在 (0,1)。
# 关于形状是 [4221, 2000]: 模型要重建的是输入矩阵 count_X 的同形状，即每个细胞 × 每个基因一个标量。。ZINB 又不止一个标量（不像普通自编码器只输出一个重构值），而是输出"均值、离散度、零概率"这组参数来定义一个概率分布。所以三个头各自都要吐出 (4221, 2000)

torch.Size([4221, 32])
torch.Size([4221, 128])
torch.Size([4221, 2000])
torch.Size([4221, 2000])
torch.Size([4221, 2000])


### 损失与反传（DSSC.py:166-170）
1. loss_zinb = ZINBLoss(x=X_raw_v, mean, disp, pi, scale_factor=size_factor_v)（layers.py:299-318）：
- mean = mean * scale_factor（把表达均值还原到 count 尺度）；
- 对原始 counts 计算零膨胀负二项负对数似然，x≈0 用 zero_case、否则 nb_case，最后 mean 归约。
2. loss.backward() + optim.Adam(amsgrad=True).step()。
3. 重复 train_iter=200 次；每次还额外做一次 eval 前向仅用于打印（DSSC.py:173-178，日志里的 Iteration:i, ZINB loss:...）。
到这里 adata.X/count_X/size_factors/A_n 的使命：把 counts 的 ZINB 结构（尤其零膨胀）压进 32 维空间隐表示 z，并让编码器学到空间图上的表示。

In [14]:
# 步骤 1
# 拿"模型预测的分布参数"和"真实原始 counts"算出一个标量损失，即所有 (细胞, 基因) 上负对数似然 -log P(x | ZINB参数) 的平均
loss_zinb = model.zinb_loss(x=X_raw_v, mean=mean_tensor, disp=disp_tensor, pi=pi_tensor, scale_factor=size_factor_v)
loss = loss_zinb
#! 把模型所有参数上一次迭代残留的 .grad 清零。因为 PyTorch 的梯度默认是累加的，不清零的话这一步算出的梯度会叠加上一轮的旧梯度，导致更新错误。它必须在 loss.backward() 之前执行
model.zero_grad()

In [17]:
loss

tensor(0.2225, device='cuda:0', grad_fn=<MeanBackward0>)

In [ ]:
# 步骤 2
loss.backward() # 反向传播，求出 loss 对每个可训练参数的梯度，并写入各参数的 .grad。只算梯度，不改参数
optim_adam.step() # Adam 优化器读取每个参数的 .grad，按 Adam 规则（带一阶/二阶动量、amsgrad=True）算出更新量，执行 param ← param − lr · 更新量。这一步真正改动权重，也就是"梯度下降"那一下

In [ ]:
# 步骤 3
if True:
    model.eval() # 切换到评估模式。这一步不是装饰，它会改变 BatchNorm 的行为（用 running 统计量，而不是当前 batch 统计量）
    loss_zinb_val = 0
    # 再做一次 aeForward + zinb_loss：重新前向一次、重新算一遍损失，然后打印 loss/num
    _, mean_tensor, disp_tensor, pi_tensor = model.aeForward(G_v, X_v) 
    loss_zinb = model.zinb_loss(x=X_raw_v, mean=mean_tensor, disp=disp_tensor, pi=pi_tensor, scale_factor=size_factor_v)
    loss_zinb_val = loss_zinb.data
    print('Iteration:{}, ZINB loss:{:.8f}'.format(1, loss_zinb_val/num))
#? 为什么不直接打印 169 行的 loss？
# 时机：169 行的 loss 是在 optim_adam.step() 之前算的，对应的是本轮更新前的旧参数；而 172–178 行是在 step 之后重新前向，反映的是刚更新完的新参数。想监控"更新后模型好不好"就得重算
# 模式：日志想在 eval() 模式下测（确定性的 BatchNorm），而训练前向是在 train() 模式下做的，两者数值本就不同

Iteration:1, ZINB loss:0.00005950


根据 DSSC.py: 163，model.train() 后流程需要循环执行 train_iter（200）次，但本次仅为测试，故只运行一次，接下来进入聚类

## 2. 进入聚类 fit（run_DSSC.py:151 → DSSC.py:204-248）

In [ ]:
# run_DSSC 中步骤
y_pred, final_loss, epoch = model.fit(X=adata.X, X_raw = count_X, X_sf=adata.obs.size_factors, A = A, A_n = A_n,
        n_clusters = n_clusters, num_epochs=400, y=y, n_ml = 4000, n_cl = 4000,
        ml_ind1=ml_ind1, ml_ind2=ml_ind2, cl_ind1=cl_ind1, cl_ind2=cl_ind2, ml_p=0.5, cl_p=1,
        p_=p_, lr = 0.001, update_interval=1, tol=0.001)

Initializing cluster centers with kmeans
Initializing kmeans: ACC= 0.6814, NMI= 0.5752, ARI= 0.4660


/hdd2/liuzy/Projects/IntensiveStudy4Lin/DSSC/src/utils.py:94: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to True or False to avoid this warning.
  l_mode = stats.mode(labels).mode[0]


Initializing kmeans KNN ACC= 0.9559
Constraints summary: ML=4000
Constraints summary: CL=4000
DEC Clustering   1: ACC= 0.6814, NMI= 0.5752, ARI= 0.4660
Clustering Iteration:1, Total loss:0.00024343, ZINB loss:0.00005298, Cluster loss:0.00005434, ML loss:0.00013612
DEC Clustering   2: ACC= 0.6603, NMI= 0.5669, ARI= 0.4591
Clustering Iteration:2, Total loss:0.00009754, ZINB loss:0.00005266, Cluster loss:0.00004008, CL loss:0.00000480
DEC Clustering   3: ACC= 0.6494, NMI= 0.5637, ARI= 0.4589
Clustering Iteration:3, Total loss:0.00021723, ZINB loss:0.00005281, Cluster loss:0.00003727, ML loss:0.00012715
DEC Clustering   4: ACC= 0.6588, NMI= 0.5684, ARI= 0.4601
Clustering Iteration:4, Total loss:0.00009799, ZINB loss:0.00005245, Cluster loss:0.00004261, CL loss:0.00000292
DEC Clustering   5: ACC= 0.6750, NMI= 0.5723, ARI= 0.4655
Clustering Iteration:5, Total loss:0.00017781, ZINB loss:0.00005202, Cluster loss:0.00004573, ML loss:0.00008007
DEC Clustering   6: ACC= 0.6764, NMI= 0.5718, ARI= 

In [19]:
np.save("data/y_pred.npy", y_pred)
np.savez("data/fit_result.npz", final_loss=float(final_loss), epoch=int(epoch))

config = dict(input_dim=adata.n_vars, encodeLayer=[128], decodeLayer=[128],
              encodeHead=3, z_dim=32, encodeConcat=False)
torch.save({"state_dict": model.state_dict(), "config": config}, "data/model_final.pt")

### 重新装载 + KMeans 初始化（DSSC.py:209-233）

预训练（run_DSSC.py:147）只是让编码器学到好的隐表示，fit 才真正把细胞分到各个空间域/簇并输出标签。
1. 再次 torch.tensor(adata.X)、重建 dgl 图 G（DSSC.py:209-211）。
2. self.mu = Parameter(7, 32)（DSSC.py:221，n_clusters=7 来自节点 1 的 Y）。
3. Zdata = encodeBatch(X, A_n)（DSSC.py:225）→ 用干净 X 过编码器得到 (4221,32) 隐表示。
4. KMeans(7, n_init=20).fit_predict(Zdata)（DSSC.py:224-226）；簇心写入 self.mu（DSSC.py:228）。
5. y（金标准）在此只用于打印初始 ACC/NMI/ARI（DSSC.py:230-233），不参与梯度。

In [23]:
# y_pred, final_loss, epoch = model.fit(X=adata.X, X_raw = count_X, X_sf=adata.obs.size_factors, A = A, A_n = A_n,
#         n_clusters = n_clusters, num_epochs=400, y=y, n_ml = 4000, n_cl = 4000,
#         ml_ind1=ml_ind1, ml_ind2=ml_ind2, cl_ind1=cl_ind1, cl_ind2=cl_ind2, ml_p=0.5, cl_p=1,
#         p_=p_, lr = 0.001, update_interval=1, tol=0.001)

# 步骤 1（DSSC.py:209-211）
# 同上，类型/格式转换。把 numpy 的 X、X_raw（原始 counts）、size_factor 转成 torch.tensor；
# 并用 dgl.from_scipy(A_n) 把归一化邻接 A_n 建成 DGL 图 G，再把 X 设为图节点特征 G.ndata['feat']
# 这样图注意力编码器才能做消息传递
X = torch.tensor(adata.X, dtype=torch.float32)
G = dgl.from_scipy(A_n)
G.ndata['feat'] = X
X_raw = torch.tensor(count_X, dtype=torch.float32)
X_sf = torch.tensor(adata.obs.size_factors, dtype=torch.float32)
# 把张量包成 Variable 并搬到 GPU
X_v = Variable(X).to("cuda")
G_v = G.to("cuda")
X_raw_v = Variable(X_raw).to("cuda")
X_sf_v = Variable(X_sf).to("cuda")

In [ ]:
# 步骤 2


(4221, 2000)

### ML/CL 索引池抽样（DSSC.py:236-248）

- ml_pool1/2 = ml_ind1/2（26203 对），cl_pool1/2 = cl_ind1/2（31828 对）——就是节点 1 从两个 txt 读入、减 1 后的数组。
- 从各自池中随机无放回抽 n_ml=4000、n_cl=4000 对（DSSC.py:240-247），打印 Constraints summary: ML/CL=4000。
- 计算 batch 数备用（DSSC.py:250-255）。

此处 ml/cl 才第一次真正“接入”模型，但它们仍只是索引；真正作用在 soft_assign 产生的软分配 q 上要等聚类循环里（DSSC.py:313-315 / 328-330）。

### 每轮开头的自训练（soft assign / target distribution, DSSC.py:263-270）
1. Zdata = self.encodeBatch(X, A_n)（DSSC.py:266）
- 用干净、无噪的 adata.X（4221×2000）过一次编码器 → 隐空间 (4221, 32)。
2. q = self.soft_assign(Zdata)（DSSC.py:267 → DSSC.py:198-202）
- 计算每个细胞到 7 个簇心 self.mu 的平方距离，套 Student-t 核：
q_ij = 1 / (1 + ||z_i − μ_j||² / α)，α = self.alpha = 1.0（DSSC.py:83）；
- 再按行归一化，得到 q 形状 (4221, 7)，即“软聚类概率”。簇心 self.mu 在此第一次参与运算。
3. p = self.target_distribution(q)（DSSC.py:268 → DSSC.py:180-182）
- p = q² / Σ_i q（按簇列求和）后再按行归一化；
- 这是 DEC 的自训练目标：对 q 锐化，得到 (4221, 7) 的“目标分布”。
4. self.y_pred = argmax(q)（DSSC.py:270）→ 本轮硬标签 (4221,)，并打印 ACC/NMI/ARI（DSSC.py:272-275，仅用 y 评估）。

### 收敛判据与掩码重采样（DSSC.py:277-294）
- delta_label = 本轮与上轮 y_pred 不同的比例；epoch>0 且 < tol(0.001) 则提前停止（DSSC.py:277-282）。第 0 轮不会停。
- epoch%1==0，所以 ml_ind1/2、cl_ind1/2 每轮都从 4000 的池里重新抽 4000 对（DSSC.py:285-294）。

### 更新聚类（第一次真正改动网络，DSSC.py:296-307）
1. _, qbatch, mean, disp, pi = self.aeForward2(G_v, X_v)（DSSC.py:299 → DSSC.py:123-133）
- aeForward2 与预训练的 aeForward 基本相同，但额外用干净输入算一次 z0 并返回 q = soft_assign(z0)：即同一次调用里“带噪输入做重建、干净输入做聚类”。
2. loss_cluster = self.cluster_loss(target=p, qbatch)（DSSC.py:300 → DSSC.py:184-188）
- KL 散度：mean_i Σ_k p_ik · log(p_ik / (q_ik+ε))，驱动 q 逼近锐化目标 p。
3. loss = loss_zinb + γ · loss_cluster，γ = 0.01（DSSC.py:301-302）；backward() + clip_grad_norm_(self.mu, 1) + step()（DSSC.py:303-306）。
- 至此 self.mu（簇心）第一次被梯度更新。

### 单约束交替优化，用 epoch%2 决定用哪一种（ML / CL），本回合 epoch=0 用 ML（DSSC.py:309-338）
- 由于第一次进入该模块时 epoch=0 为偶数 → 走 ML 分支（DSSC.py:310-323）：
- 再前向一次得到新的 qbatch（DSSC.py:311-312）；
- ml_q1 = qbatch[ml_ind1]、ml_q2 = qbatch[ml_ind2]，各 (4000, 7)——这是 ml 文件第一次真正作用于模型；
- ml_loss = mean(-log(Σ_k q1_k·q2_k))（DSSC.py:315 → DSSC.py:192），即让同层细胞的软分配重叠、互相靠拢；
- loss = ml_p · ml_loss + loss_zinb，ml_p = weight_ml = 0.5（DSSC.py:316）；反传 + 裁剪 + 更新（DSSC.py:317-320）。

注意：每个 epoch 实际做了 3 次前向（encodeBatch、aeForward2 聚类、aeForward2 约束；而 aeForward2 内部又各含 2 次编码），且 ml/cl 各 4000 对是全批量参与，batch_size=256 在聚类阶段并未被使用（DSSC.py:250-255 的 batch 数算了但没用）。

至此，一个 epoch（epoch = 0）结束，回到 DSSC.py:262，接下来 epoch = 1。

### epoch=1 的前半段（与 epoch 0 相同，但数据已变）（DSSC.py:263-306：以更新过一次的 self.mu 重新走一遍）
1. Zdata = encodeBatch(X, A_n) → q = soft_assign → p = target_distribution → y_pred = argmax(q)（DSSC.py:266-270）。
2. 计算 delta_label 并判断早停（DSSC.py:277-282）——本 log 的 run1 在第 34 个 epoch 才满足，故此处不停。
3. epoch%1==0：重新抽 ml_ind1/2、cl_ind1/2 各 4000 对（DSSC.py:285-294）。
4. loss = loss_zinb + γ·cluster_loss(p, qbatch)，更新 mu（DSSC.py:299-307）。

### 单约束交替优化，本回合 epoch=1 用 CL（DSSC.py:309-338）
因为 epoch=1 是奇数，走 else：
1. 再前向一次：_, qbatch, mean, disp, pi = self.aeForward2(G_v, X_v)（DSSC.py:326），得到新的软分配 qbatch (4221,7)。
2. cl 文件在此接管：
- cl_q1 = qbatch[cl_ind1]、cl_q2 = qbatch[cl_ind2]，各 (4000, 7)（DSSC.py:328-329）；
- cl_ind1/2 是节点 1 从 sample_151507_clFromMarks_test.txt 读入的细胞索引对（减 1 后），此处正是它们第一次参与损失。
3. cl_loss = self.pairwise_loss(cl_q1, cl_q2, "CL")（DSSC.py:330 → DSSC.py:195）：
mean(-log(1 − Σ_k q1_k·q2_k))——两个异层细胞的软分配重叠度越高，惩罚越大，迫使它们分到不同簇。
4. loss = cl_p·cl_loss + loss_zinb，cl_p = weight_cl = 1.0（DSSC.py:331）；backward + clip_grad_norm_(mu,1) + step（DSSC.py:332-335）。

### ML / CL 的交替规律（epoch%2）
- 偶数 epoch（0,2,4,…）→ ML：ml_p=0.5，把同层细胞拉近；
- 奇数 epoch（1,3,5,…）→ CL：cl_p=1.0，把异层细胞推远；
- 每轮开头还叠加 γ=0.01 的 cluster_loss 自训练，以及始终存在的 loss_zinb 重建项。
也就是说，mu/编码器每轮接受 2 次参数更新：先由 (ZINB + 0.01·聚类) 更新，再由 (0.5·ML 或 1.0·CL + ZINB) 更新；而 ml/cl 各 4000 对每轮都会重新采样。